# Claude Developer Platform — Build-Along

**Level:** 300 (Intermediate) | **Model:** Claude Sonnet 5 (`claude-sonnet-5`)

## What We're Building

A multi-tool **support ticket agent** that reads ticket details, searches a knowledge base, and produces a structured resolution — using the Claude API directly, no framework.

**Customer scenario:** TechFlow (mid-market B2B SaaS, 500+ tickets/day) wants to automate Tier 1 triage so human agents can focus on complex escalations.

## Learning Objectives

1. Implement a multi-tool agentic loop (`while stop_reason == "tool_use"`)
2. Compose structured outputs with tool use in a single agent
3. Integrate adaptive thinking with `output_config.effort` to control reasoning depth
4. Stream thinking, tool calls, and responses in real-time

## Where Does This Fit?

There are several ways to build with Claude. This session focuses on the **Messages API** — the lowest level, giving you full control over the request/response cycle:

| Surface | What It Is | When to Use It |
|---------|-----------|----------------|
| **Messages API** ← *this session* | Direct HTTP/SDK calls to Claude | Full control over agentic loops, custom orchestration, production backends |
| **Agent SDK** | Python framework with built-in tool dispatch | Rapid agent prototyping, when you want orchestration handled for you |
| **Claude Code** | CLI-based coding agent | Developer productivity, repo-level tasks, interactive coding |
| **claude.ai / Claude for Enterprise** | Chat interface with Projects, MCP | End-user workflows, enterprise knowledge work, non-developer use cases |

Today you build the raw loop. Understanding this makes everything above it clearer.

---

**Prepared for Partner Basecamp participants.** Not for reproduction or redistribution as training material — you're free to apply these patterns in your own client work.

## Setup

Run the cells below to connect to Claude and load mock data.

- **Your API key** lives in a `.env` file at the repo root (gitignored) — `basecamp_setup.py` puts it there.
- **Look for ✏️ YOUR TURN cells** — they mark exactly where you write code today.
- **API cells take time** — an agent run with thinking can take 30–90 seconds. The `[*]` next to a cell means it's still working.


### Setup — connect to Claude

Run the next cell. It reads your key from the `.env` file at the repo root and checks it with a real call — you're ready when you see the green **✓ verified** banner. Edited `.env`? Just run the cell again. Red or yellow banner? Do what it says and run the cell again.

**First time on this machine?** From the repo root (the folder that contains `day1` and `day2`), run `python3 basecamp_setup.py` once (Windows: `py basecamp_setup.py`; see `SETUP.md`), then pick the **Anthropic Basecamp (.venv)** kernel (top right).

In [ ]:
# ── Setup — connect to Claude ──
# One-time machine setup is basecamp_setup.py at the repo root: run it once, then pick the
# "Anthropic Basecamp (.venv)" kernel. This cell reads your key from .env and checks it with a real
# call. Edited .env? Just run this cell again.
import json
import pathlib
import sys
import time

_root = next((d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (d / "basecamp_setup.py").is_file()), None)
if _root is None:
    raise SystemExit("Can't find basecamp_setup.py — open this notebook from inside the "
                     "Basecamp-Exercises folder (see SETUP.md).")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from basecamp_setup import connect

claude = connect()
client = claude.client
MODEL, FAST_MODEL, BIG_MODEL = claude.model, claude.fast_model, claude.big_model


In [ ]:
# ── Sample Ticket Data ──

TICKETS = {
    "TKT-1042": {
        "id": "TKT-1042", "customer": "Acme Corp", "priority": "high",
        "product_area": "billing",
        "description": "We were charged twice for our March invoice. Invoice #INV-2024-0342 shows $4,500 but our bank shows two identical charges on March 3rd. Need immediate refund of the duplicate charge.",
        "status": "open"
    },
    "TKT-1043": {
        "id": "TKT-1043", "customer": "DataFlow Inc", "priority": "medium",
        "product_area": "api",
        "description": "Our webhook endpoint stopped receiving events after we rotated API keys yesterday. We've verified the new key works for REST calls but webhooks are still failing. Getting 401 errors in the webhook logs.",
        "status": "open"
    },
    "TKT-1044": {
        "id": "TKT-1044", "customer": "CloudScale Ltd", "priority": "low",
        "product_area": "feature_request",
        "description": "Would love to see bulk export functionality in the dashboard. Currently we have to export reports one at a time which is painful when we need quarterly summaries across 50+ projects.",
        "status": "open"
    },
    "TKT-1045": {
        "id": "TKT-1045", "customer": "SecureNet Systems", "priority": "critical",
        "product_area": "account",
        "description": "Our admin account (admin@securenet.io) is locked out after failed MFA attempts. We have 47 team members who can't access the platform because SSO is tied to this admin account. This is blocking all work.",
        "status": "open"
    },
    "TKT-1046": {
        "id": "TKT-1046", "customer": "MedTech Solutions", "priority": "high",
        "product_area": "api",
        "description": "Our production integration started returning intermittent 500 errors around 2am last night. About 15% of API calls are failing. We haven't changed anything on our end. Errors seem random - sometimes the same request works on retry. Our team in Singapore is blocked and we need this resolved ASAP.",
        "status": "open"
    },
}

KB_ARTICLES = {
    "KB-001": {"title": "Processing Duplicate Payment Refunds", "content": "For duplicate charges: 1) Verify the duplicate in the billing system, 2) Issue refund through the payment processor (takes 3-5 business days), 3) Send confirmation email with refund reference number. Escalate if amount exceeds $10,000."},
    "KB-002": {"title": "Webhook Authentication After Key Rotation", "content": "When API keys are rotated, webhook signing secrets must also be updated. Go to Settings > Webhooks > Edit endpoint, and regenerate the signing secret. The old secret is invalidated immediately on key rotation. Common mistake: rotating the API key but not the webhook signing secret."},
    "KB-003": {"title": "Bulk Export Feature (Roadmap)", "content": "Bulk export is on the Q3 roadmap. Workaround: Use the REST API's /reports/export endpoint with date range parameters to programmatically export multiple reports. See API docs for batch export examples."},
    "KB-004": {"title": "Admin Account Lockout Recovery", "content": "For locked admin accounts: 1) Verify identity through the secondary email on file, 2) Reset MFA through the admin recovery flow at /admin/recover, 3) Temporary access can be granted through support-level override (requires manager approval). Critical: If SSO is blocked, enable the bypass login at /login/direct for affected users."},
    "KB-005": {"title": "API Rate Limiting Best Practices", "content": "Default rate limits: 100 requests/minute for standard plans, 1000/minute for enterprise. Use exponential backoff with jitter for retries. Monitor usage via the X-RateLimit headers in responses."},
    "KB-006": {"title": "Invoice Discrepancy Resolution", "content": "For billing discrepancies: Check the billing audit log for the account, compare with payment processor records, and verify no pending transactions. Contact finance team for adjustments over $5,000."},
    "KB-007": {"title": "Intermittent 500 Errors Troubleshooting", "content": "For intermittent server errors: 1) Check the status page for known outages, 2) Review rate limit headers - 429s can masquerade as 500s behind load balancers, 3) Check if errors correlate with payload size or specific endpoints, 4) Enable request ID logging and contact support with specific request IDs for investigation. If >10% error rate persists for >1 hour, escalate to engineering."},
}

def get_ticket(ticket_id: str) -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        return json.dumps(ticket)
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

def search_kb(query: str) -> str:
    query_lower = query.lower()
    results = []
    for article_id, article in KB_ARTICLES.items():
        if any(word in article["title"].lower() or word in article["content"].lower()
               for word in query_lower.split() if len(word) > 2):
            results.append({"id": article_id, **article})
    if not results:
        results = [{"id": "KB-000", "title": "No matches found", "content": "No relevant articles found. Consider escalating to Tier 2 support."}]
    return json.dumps(results[:3])

def resolve_ticket(ticket_id: str, resolution: str, status: str = "resolved") -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        ticket["status"] = status
        ticket["resolution"] = resolution
        return json.dumps({"success": True, "ticket_id": ticket_id, "new_status": status})
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

TOOL_FUNCTIONS = {"get_ticket": get_ticket, "search_kb": search_kb, "resolve_ticket": resolve_ticket}

def execute_tool(name: str, input_data: dict) -> str:
    func = TOOL_FUNCTIONS.get(name)
    if func:
        return func(**input_data)
    return json.dumps({"error": f"Unknown tool: {name}"})

print("Mock tools and sample data loaded!")
print(f"   Available tickets: {', '.join(TICKETS.keys())}")
print(f"   Knowledge base articles: {len(KB_ARTICLES)}")

---# Part 1: Multi-Tool Agentic Loop## The Problem: Manual Ticket TriageTechFlow's Tier 1 support team currently handles each ticket manually through a repetitive 4-step process:1. **Look up the customer** — pull account details, tier, history2. **Search the knowledge base** — find relevant articles or procedures3. **Categorize the issue** — billing, technical, account, or feature request4. **Draft a resolution** — write instructions, set status, track escalations**Scale:** 500+ tickets per day × 8 minutes each = **67 person-hours daily** on repetitive triage. The team wants Claude to handle this autonomously so human agents can focus on complex escalations that require judgment, empathy, and creative problem-solving.---## Why an Agentic Loop?A single Claude API call can't solve a support ticket — the model needs to:- First **look up** the ticket to see what the problem is- Then **search** for solutions based on what it found- Finally **resolve** the ticket with the right informationThis requires **multiple sequential API calls** where Claude:1. Decides which tool to call based on the current state2. Receives the tool results3. Decides the next action (call another tool or provide a final answer)This is the **agentic loop pattern**: `while response.stop_reason == "tool_use"`, extract tool calls → execute them → send results back → repeat. Claude orchestrates the workflow; your code just handles the plumbing.---## What We're BuildingA **three-tool autonomous agent** with no framework — just the Claude API, Python, and a loop:```┌─────────────────────────────────────────────────────────┐│  User: "Resolve ticket TKT-1042"                        │└─────────────────────────────────────────────────────────┘                        ↓         ┌──────────────────────────────┐         │   Claude Messages API        │         │   + Tool Definitions         │         └──────────────────────────────┘                        ↓              [tool_use: get_ticket]                        ↓         ┌──────────────────────────────┐         │  Execute: get_ticket()       │         │  → Returns ticket JSON       │         └──────────────────────────────┘                        ↓         ┌──────────────────────────────┐         │   Send result back to Claude │         └──────────────────────────────┘                        ↓            [tool_use: search_kb]                        ↓         ┌──────────────────────────────┐         │  Execute: search_kb()        │         │  → Returns KB articles       │         └──────────────────────────────┘                        ↓         ┌──────────────────────────────┐         │   Send result back to Claude │         └──────────────────────────────┘                        ↓           [tool_use: resolve_ticket]                        ↓         ┌──────────────────────────────┐         │  Execute: resolve_ticket()   │         │  → Returns success           │         └──────────────────────────────┘                        ↓         ┌──────────────────────────────┐         │   Final Response             │         │   stop_reason: "end_turn"    │         └──────────────────────────────┘```The loop continues **until Claude stops requesting tools**. This makes the agent truly autonomous — you don't hard-code the sequence; Claude decides it dynamically based on what it learns at each step.---## Why No Framework?Frameworks like LangChain or our own Agent SDK abstract away the loop, which is convenient for rapid prototyping. But understanding the **raw pattern** is critical because:1. **Debugging** — When your agent misbehaves, you need to inspect the exact messages and tool calls2. **Customization** — Production systems need custom retry logic, cost tracking, latency optimization3. **Portability** — This pattern works in any language and with any tool execution environment4. **Control** — You decide exactly when to call the API, how to batch tool results, when to interruptOnce you understand this, frameworks become **convenience wrappers** you can reason about — not magic boxes you depend on blindly.---## Key Concept: Adaptive ThinkingClaude Sonnet 5 introduces **adaptive thinking** — the model automatically decides how much internal reasoning to do based on task complexity. You enable it with:```pythonthinking={"type": "adaptive"}```**What it does:**- **Simple tickets** (password reset, duplicate charge) → brief reasoning, fast execution- **Complex tickets** (intermittent API errors, security escalations) → deep reasoning, thorough analysis- **Thinking traces** are returned in `thinking` content blocks, giving you visibility into *why* Claude made each decision**Why it matters for this agent:**- No need to manually tune "chain-of-thought" prompts — Claude auto-calibrates- Thinking traces make the agent **auditable** (critical for support workflows where wrong decisions have customer impact)- You can control reasoning depth with `output_config.effort` (we'll explore this in Part 2)---## The Pattern You'll LearnThe core agentic loop has three parts:1. **Tool Definition** — JSON schemas that tell Claude what tools exist and how to call them2. **Loop Orchestration** — `while stop_reason == "tool_use"`, extract calls, execute, send back results3. **Result Extraction** — Parse Claude's final response (text or structured JSON)This pattern powers everything from simple ticket resolution to complex multi-agent systems. Master it here, apply it everywhere.Let's build it.

## Part 2: Define Tool Schemas

TechFlow's support agent needs access to three systems: the ticketing platform (to look up details), the knowledge base (to find solutions), and the resolution engine (to close tickets). Each of these becomes a tool schema that tells Claude what's available and how to call it.

The `description` field is critical — it's how Claude decides *which* tool to use for a given step. Vague descriptions lead to wrong tool selection.

> 📖 **Reference:** [Tool use documentation](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)

### ✏️ YOUR TURN — complete the `search_kb` and `resolve_ticket` tool schemas

`get_ticket` is already filled in as your reference. Complete the empty `description` strings and the `status` enum wherever you see `<-- fill this in`.

In [ ]:
# TODO: Define tool schemas for search_kb and resolve_ticket# Each tool needs: name, description, input_schema (with properties and required)tools = [    # ✅ Example: get_ticket is done — use this as your reference for the next two    {        "name": "get_ticket",        "description": "Retrieve full details for a support ticket by its ID, including customer, priority, product area, and description.",        "input_schema": {            "type": "object",            "properties": {                "ticket_id": {"type": "string", "description": "The ticket ID, e.g. TKT-1042"}            },            "required": ["ticket_id"]        }    },    # ✅ COMPLETED: search_kb    {        "name": "search_kb",        "description": "Search the knowledge base for articles, procedures, and solutions related to a support issue. Use keywords from the ticket description to find relevant documentation.",        "input_schema": {            "type": "object",            "properties": {                "query": {"type": "string", "description": "Search keywords or phrases describing the issue (e.g., 'duplicate payment refund', 'webhook authentication', 'API 500 errors')"}            },            "required": ["query"]        }    },    # ✅ COMPLETED: resolve_ticket    {        "name": "resolve_ticket",        "description": "Mark a support ticket as resolved, escalated, or pending with a detailed resolution message. This should be called after gathering all necessary information and determining the solution.",        "input_schema": {            "type": "object",            "properties": {                "ticket_id": {"type": "string", "description": "The ticket ID to resolve"},                "resolution": {"type": "string", "description": "Detailed resolution text with specific steps, timelines, and next actions"},                "status": {"type": "string", "enum": ["resolved", "escalated", "pending"], "description": "The final status of the ticket"}            },            "required": ["ticket_id", "resolution", "status"]        }    }]print(f"Defined {len(tools)} tool schemas: {[t['name'] for t in tools]}")

## Part 3: Build the Agentic Loop

Here's the core automation: when a ticket comes in, Claude should look it up, search for a solution, and resolve it — all without human intervention. The agentic loop makes this possible: `while response.stop_reason == "tool_use"`, extract tool calls, execute them, append results, and call the API again. Claude decides the sequence; your code just orchestrates.

**Key:** Pass `response.content` back as-is — it may contain thinking blocks alongside tool_use blocks. Claude Sonnet 5 uses `thinking={"type": "adaptive"}` on every call.

> 📖 **Reference:** [Agentic tool use patterns](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/agentic-tool-use)

### ✏️ YOUR TURN — build the agentic loop

Fill in every `___` blank in `run_agent()` below: the loop condition, the tool-result wiring, and the follow-up API call.

In [ ]:
SYSTEM_PROMPT = """You are a Tier 1 support agent for TechFlow, a B2B SaaS platform that provides project management and team collaboration tools to mid-market companies.## Your RoleYou handle incoming support tickets by investigating issues, finding solutions in the knowledge base, and resolving tickets with clear, actionable guidance.## Process1. ALWAYS look up the ticket first to understand the full context2. Search the knowledge base for relevant solutions and procedures3. Resolve the ticket with a detailed resolution that includes specific next steps## Guidelines- Be thorough: always search the KB before resolving, even if the issue seems straightforward- Be specific: include exact steps, links, and timeframes in resolutions- Escalate when needed: if confidence is low or the issue requires privileged access, mark for escalation- Categorize accurately: billing, technical, account, or feature_request## Escalation Criteria- Financial issues over $10,000- Security-related account compromises- Issues requiring engineering intervention- Customers with Enterprise SLA (response within 1 hour)## TechFlow Product Tiers- Starter ($29/user/month): Basic project management, 5GB storage, email support, 5 projects max, community forums- Professional ($79/user/month): Advanced analytics, 100GB storage, priority support, API access, unlimited projects, custom fields, Gantt charts, time tracking- Enterprise (custom pricing): SSO/SAML, unlimited storage, dedicated CSM, custom integrations, SLA guarantees, audit logs, advanced security, custom branding, priority API rate limits## Common Issue Categories and Routing- Billing: Invoice discrepancies, payment failures, plan changes, refund requests, subscription cancellations, proration questions- Technical: API errors, integration issues, webhook failures, performance problems, data export issues, browser compatibility- Account: Login issues, MFA problems, SSO configuration, permission changes, team management, user provisioning- Feature Requests: Product feedback, roadmap inquiries, workaround requests, beta access requests## Response TemplatesWhen resolving billing issues, always include: transaction ID, refund timeline, and confirmation email details.When resolving technical issues, always include: steps to reproduce, workaround if available, and engineering ticket number if escalated.When resolving account issues, always include: security verification steps taken and any temporary access granted.## SLA Requirements- Starter: 24-hour response time, business hours only- Professional: 4-hour response time, extended hours (6am-10pm)- Enterprise: 1-hour response time, 24/7 support, dedicated Slack channel## ToneProfessional, empathetic, and solution-oriented. Acknowledge the customer frustration before jumping to the solution. Use the customer name when available. Reference the specific product tier for relevant guidance."""# ✅ COMPLETED: Implement run_agent(user_message)def run_agent(user_message: str):    """Run the support ticket agent."""    messages = [{"role": "user", "content": user_message}]    response = client.messages.create(        model=MODEL,        max_tokens=32000,        system=SYSTEM_PROMPT,        tools=tools,        thinking={"type": "adaptive"},        messages=messages    )    # ✅ Loop while Claude still wants to use tools    while response.stop_reason == "tool_use":        tool_results = []        for block in response.content:            if block.type == "tool_use":                result = execute_tool(block.name, block.input)                tool_results.append({                    "type": "tool_result",                    "tool_use_id": block.id,  # ✅ block.id links this result back to the tool call                    "content": str(result)                })        # ✅ Append assistant turn + tool results back to messages        # Key: pass ALL content blocks (including any thinking blocks) back — not just text        messages.append({"role": "assistant", "content": response.content})        messages.append({"role": "user", "content": tool_results})        # ✅ Call the API again with the updated messages        response = client.messages.create(            model=MODEL,            max_tokens=32000,            system=SYSTEM_PROMPT,            tools=tools,            thinking={"type": "adaptive"},            messages=messages        )    return response# Test it!# response = run_agent("Resolve ticket TKT-1042")# for block in response.content:#     if block.type == "text" and block.text.strip():#         print(f"\n Final response:\n{block.text}")

## Part 4: Add Structured Output## The Problem: Machines Need StructureYour basic agent works great for humans reading ticket resolutions in a dashboard. But TechFlow's platform has **downstream systems** that need to consume these resolutions programmatically:1. **Ticketing Database** — Updates status, categories, timestamps2. **Analytics Dashboard** — Tracks resolution rates by category3. **Escalation Router** — Automatically forwards tickets flagged with `escalation_needed: true`4. **SLA Monitor** — Measures confidence levels and time-to-resolution5. **Knowledge Base Updater** — Identifies patterns in solutions for new KB articles**Free-text responses break all of these integrations.** You'd need fragile regex parsing, error-prone LLM-based extraction, or manual human review.---## The Solution: Structured Outputs (JSON Mode)Claude's **structured outputs** feature constrains the model's text response to match a JSON schema. Instead of:```"I've investigated TKT-1042 and found it's a duplicate billing issue. The customer should receive a refund in 3-5 business days..."```You get:```json{  "diagnosis": "Duplicate charge on March invoice INV-2024-0342",  "solution_steps": [    "Verify duplicate in billing system",    "Issue refund through payment processor (3-5 business days)",    "Send confirmation email with refund reference"  ],  "confidence": "high",  "escalation_needed": false,  "category": "billing"}```**Every field is guaranteed** — no null checks, no parsing errors, no missing data. Your downstream systems can deserialize JSON and immediately act on the structured fields.---## How It Works: `output_config.format`You define a JSON schema and pass it via `output_config`:```pythonRESOLUTION_SCHEMA = {    "type": "json_schema",    "schema": {        "type": "object",        "properties": {            "diagnosis": {"type": "string"},            "solution_steps": {"type": "array", "items": {"type": "string"}},            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},            "escalation_needed": {"type": "boolean"},            "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}        },        "required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"]    }}response = client.messages.create(    model=MODEL,    output_config={"format": RESOLUTION_SCHEMA},  # ✅ Constrains output to this schema    ...)```Claude will **only** return valid JSON matching this schema. No extra text, no markdown formatting, just pure JSON.---## Critical Constraint: Tools vs. Structured OutputHere's the catch: **`output_config.format` applies to ALL text output from Claude.**When you set `format`, Claude can no longer:- Return free-form text explanations- Use tools (because tool responses aren't JSON-schema-constrained)- Mix thinking blocks with text**This means:**❌ You **cannot** use `output_config.format` during the agentic tool loop✅ You **can** use it on the *final* call after tools are done---## The Two-Phase Pattern```Phase 1: Agentic Tool Loop (NO format constraint)┌────────────────────────────────────────┐│  while stop_reason == "tool_use":      ││    - Claude uses tools freely          ││    - No format constraint              ││    - Thinking + tool_use blocks        │└────────────────────────────────────────┘                 ↓         (loop completes)                 ↓Phase 2: Structured Output (WITH format constraint)┌────────────────────────────────────────┐│  Append: "Provide resolution as JSON"  ││  Call with:                            ││    output_config={"format": SCHEMA}    ││    tool_choice={"type": "none"}        ││  Returns: Pure JSON matching schema    │└────────────────────────────────────────┘```**Why `tool_choice: {"type": "none"}`?**This disables tool use on the final call. Claude already gathered all the information during Phase 1 — now we just want the structured summary. Without this, Claude might try to call tools again, which conflicts with the format constraint.---## Adaptive Thinking with Structured Output**Important:** Structured output works seamlessly with `thinking={"type": "adaptive"}`.When you combine them:```pythonresponse = client.messages.create(    model=MODEL,    thinking={"type": "adaptive"},    output_config={"format": RESOLUTION_SCHEMA},    ...)```The response will contain:```response.content = [    ThinkingBlock(thinking="Let me analyze the resolution..."),    TextBlock(text='{"diagnosis": "...", "solution_steps": [...], ...}')]```**The thinking is unrestricted** (Claude reasons naturally), but **the text block is pure JSON** matching your schema.**To extract it:**```python# Get the last text block (the JSON)text_blocks = [b for b in response.content if b.type == "text"]result = json.loads(text_blocks[-1].text)```---## Schema Design Best Practices### 1. **Use Enums for Categories**```python"category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}```✅ Forces valid categories — no typos, no variations❌ Without enum: "Billing", "billing_issue", "bill", etc.### 2. **Arrays for Step-by-Step Instructions**```python"solution_steps": {"type": "array", "items": {"type": "string"}}```✅ Makes it easy for UI to render numbered lists✅ Analytics can measure step complexity### 3. **Boolean Flags for Routing**```python"escalation_needed": {"type": "boolean"}```✅ Downstream systems can route with simple `if escalation_needed:`❌ String-based: "yes", "true", "escalate" → brittle parsing### 4. **Required Fields = No Nulls**```python"required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"]```✅ Claude **must** provide all fields — no missing data✅ Your code never needs null checks### 5. **Descriptions Guide Generation**```python"diagnosis": {"type": "string", "description": "Root cause analysis of the issue"}```✅ Claude understands what belongs in each field✅ Produces more accurate, detailed output---## Real-World ImpactTechFlow's downstream integrations now work reliably:| System | Without Structure | With Structure ||--------|------------------|----------------|| **Database** | Regex to extract status → 12% parse failures | Direct field access: `row.status = result['category']` || **Analytics** | Manual categorization → inconsistent labels | Enum categories → perfect aggregation || **Escalation Router** | Search for keywords like "escalate" | Simple boolean: `if result['escalation_needed']` || **SLA Monitor** | No confidence data available | Tracks: 78% high confidence, avg 2.3 min resolution |**Before:** 3 engineers maintaining parsing logic, weekly incidents from bad parses**After:** Zero parsing code, zero parse failures, full audit trail---## What You'll BuildYou'll extend `run_agent()` to return structured JSON:```pythonresult = run_agent_structured("Resolve ticket TKT-1042")# Guaranteed structure:print(result['category'])           # "billing"print(result['escalation_needed'])  # Falseprint(result['confidence'])         # "high"print(len(result['solution_steps'])) # 3```The agent runs the full tool loop (lookup ticket, search KB, gather info), **then** produces a machine-readable resolution on the final call.Let's implement it.

### ✏️ YOUR TURN — finish `run_agent_structured()`

The tool loop is already written for you. Fill in the two `___` blanks on the final call: `output_config` and `tool_choice`.

In [ ]:
# ✅ RESOLUTION_SCHEMA is already defined below — read it, then implement run_agent_structured()RESOLUTION_SCHEMA = {    "type": "json_schema",    "schema": {        "type": "object",        "properties": {            "diagnosis": {"type": "string", "description": "Root cause analysis of the issue"},            "solution_steps": {"type": "array", "items": {"type": "string"}, "description": "Ordered steps to resolve"},            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},            "escalation_needed": {"type": "boolean"},            "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}        },        "required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"],        "additionalProperties": False    }}def get_structured_result(response) -> dict:    """Extract the structured JSON from the last text block in the response."""    # With adaptive thinking, content may be [thinking, text] - JSON is in the last text block    text_blocks = [b for b in response.content if b.type == "text" and b.text.strip()]    if text_blocks:        return json.loads(text_blocks[-1].text)    return Nonedef run_agent_structured(user_message: str) -> dict:    """Run the agent with structured JSON output."""    # Step 1: Run the tool loop — same as run_agent(), NO output_config here    # (format constrains ALL text output, so tools won't work with it active)    messages = [{"role": "user", "content": user_message}]    response = client.messages.create(        model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,        tools=tools, thinking={"type": "adaptive"}, messages=messages    )    while response.stop_reason == "tool_use":        tool_results = []        for block in response.content:            if block.type == "tool_use":                result = execute_tool(block.name, block.input)                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})        messages.append({"role": "assistant", "content": response.content})        messages.append({"role": "user", "content": tool_results})        response = client.messages.create(            model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,            tools=tools, thinking={"type": "adaptive"}, messages=messages        )    # Step 2: One final call to get structured output    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})    # ✅ COMPLETED: fill in the two blanks    final = client.messages.create(        model=MODEL,        max_tokens=8000,        system=SYSTEM_PROMPT,        output_config={"format": RESOLUTION_SCHEMA},   # ✅ Pass RESOLUTION_SCHEMA using {"format": RESOLUTION_SCHEMA}        tool_choice={"type": "none"},     # ✅ Prevent further tool calls: {"type": "none"}        thinking={"type": "adaptive"},        messages=messages    )    return get_structured_result(final)# result = run_agent_structured("Resolve ticket TKT-1042")# print(json.dumps(result, indent=2))

---
### ✅ CHECKPOINT 1 — Working support ticket agent with structured output

You should have an agent that calls 2–3 tools per ticket and returns structured JSON.

**Verify:** `run_agent_structured("Resolve ticket TKT-1044")` → category: feature_request, suggests API workaround

---
# Part 2: Adaptive Thinking

The basic agent works, but TechFlow's support lead raises a concern: "Some tickets are straightforward — duplicate charges, password resets. Others are genuinely ambiguous. We want deeper reasoning on hard tickets without slowing down the easy ones. And when the agent gets a hard ticket wrong, we can't see *why* it made that call."

Adaptive thinking solves both problems. The `output_config.effort` parameter lets you control how deeply Claude reasons — "high" for complex or ambiguous tickets, "low" for straightforward ones. Thinking traces give you visibility into the decision process, so you can audit *why* the agent chose to escalate or which KB article it weighed most heavily.

> 📖 **Reference:** [Adaptive thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

## Cell 5: Add Effort-Level Thinking Control

Add `output_config.effort` to control how deeply Claude reasons. With high effort, Claude produces detailed thinking traces between tool calls — reasoning about what to search for, evaluating KB results, deciding whether to escalate. With low effort, it keeps reasoning brief for simple tickets. Display thinking blocks so you can see the agent's decision process.

### ✏️ YOUR TURN — implement `run_agent_thinking()`

Write the whole function this time — the comments walk you through the four steps.

In [ ]:
# ✅ COMPLETED: Add effort-level thinking control to the agentdef run_agent_thinking(user_message: str, effort: str = "high") -> dict:    """Run agent with effort-controlled adaptive thinking."""    messages = [{"role": "user", "content": user_message}]        # Step 1: Run the tool loop with thinking and effort control (but NO format constraint)    response = client.messages.create(        model=MODEL,        max_tokens=32000,        system=SYSTEM_PROMPT,        tools=tools,        thinking={"type": "adaptive"},        output_config={"effort": effort},  # Control reasoning depth        messages=messages    )        # Step 2: Display thinking blocks and execute tools in the loop    while response.stop_reason == "tool_use":        # Display thinking traces        for block in response.content:            if block.type == "thinking":                print(f"[Thinking] {block.thinking[:200]}..." if len(block.thinking) > 200 else f"[Thinking] {block.thinking}")            elif block.type == "tool_use":                print(f"[Tool] {block.name}({json.dumps(block.input)})")                # Execute tools        tool_results = []        for block in response.content:            if block.type == "tool_use":                result = execute_tool(block.name, block.input)                tool_results.append({                    "type": "tool_result",                    "tool_use_id": block.id,                    "content": str(result)                })                print(f"[Result] {result[:150]}..." if len(result) > 150 else f"[Result] {result}")                # Continue the loop        messages.append({"role": "assistant", "content": response.content})        messages.append({"role": "user", "content": tool_results})                response = client.messages.create(            model=MODEL,            max_tokens=32000,            system=SYSTEM_PROMPT,            tools=tools,            thinking={"type": "adaptive"},            output_config={"effort": effort},            messages=messages        )        # Step 3: After the tool loop ends, make a FINAL call with structured output    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})        final = client.messages.create(        model=MODEL,        max_tokens=8000,        system=SYSTEM_PROMPT,        output_config={"effort": effort, "format": RESOLUTION_SCHEMA},  # Both effort AND format        tool_choice={"type": "none"},        thinking={"type": "adaptive"},        messages=messages    )        # Step 4: Extract and return the structured result    return get_structured_result(final)

## Cell 6: Explore Adaptive Thinking in Action

MedTech Solutions (TKT-1046) is reporting intermittent 500 errors — 15% of API calls failing, no changes on their end, started at 2am. Is it rate limiting? A server-side outage? Network issues? The KB won't have a perfect match.

This is where thinking earns its keep. Run the ambiguous ticket with high effort, then compare it with low effort on the same ticket. Watch how reasoning depth changes — and ask yourself: for TechFlow's 500 tickets/day, how would you route simple vs. complex tickets to different effort levels?

In [ ]:
def _not_built_yet(fn_name, result):
    """Build-along guard: the YOUR TURN stubs return None until you implement them."""
    if result is not None:
        return False
    print(f"\n[--] {fn_name}() isn't built yet - that's the exercise, not a bug.")
    print(f"     Find the '# YOUR TURN' marker inside {fn_name}(), build it, then run this again.")
    return True

# Run the ambiguous ticket at high effort — observe the thinking traces
print("=== TKT-1046: Intermittent API Errors (ambiguous) ===\n")
result = run_agent_thinking("Resolve ticket TKT-1046", effort="high")
if _not_built_yet("run_agent_thinking", result):
    raise SystemExit(0)
print(f"\nResolution:")
print(json.dumps(result, indent=2))

# Now compare: same ticket, low effort
print(f"\n\n{'='*50}")
print("=== Same ticket, LOW effort ===")
print(f"{'='*50}\n")

for effort in ["high", "low"]:
    start = time.time()
    result = run_agent_thinking("Resolve ticket TKT-1046", effort=effort)
    elapsed = time.time() - start
    print(f"\n[effort={effort}] Confidence: {result['confidence']} | Steps: {len(result['solution_steps'])} | Escalate: {result['escalation_needed']} | Time: {elapsed:.1f}s")

---
### ✅ CHECKPOINT 2 — Agent with reasoning visibility + effort control

Effort comparison should show observable differences in reasoning depth and quality.

---
# Part 3: Streaming

TechFlow's support agents use a real-time dashboard to monitor the AI triage system. When a ticket comes in, they need to *see* the agent working — which ticket it's looking up, what it's searching for, how it's reasoning. A 15-second spinner followed by a wall of text doesn't build trust.

Streaming solves this. Replace `create()` with `stream()` and tokens arrive in real-time: thinking traces flow as the agent reasons, tool calls appear as they're made, and the structured resolution streams at the end.

> 📖 **Reference:** [Streaming messages](https://docs.anthropic.com/en/docs/build-with-claude/streaming)

## Cell 7: Streaming Agentic Loop

Replace `client.messages.create()` with `client.messages.stream()` so TechFlow's dashboard can display the agent's work in real-time. Handle the key event types: `thinking_delta` (reasoning tokens), `text_delta` (final response), and `input_json_delta` (tool arguments).

Use `stream.get_final_message()` after the stream completes to get the full response object for loop continuation.

### ✏️ YOUR TURN — implement `run_agent_streaming()`

Write the whole function — swap `create()` for `stream()` and handle the stream events listed in the comments.

In [ ]:
# ✅ COMPLETED: Build the streaming agentic loopdef run_agent_streaming(user_message: str, effort: str = "high") -> dict:    """Run agent with streaming output."""    messages = [{"role": "user", "content": user_message}]        # Step 1: Start the streaming loop    with client.messages.stream(        model=MODEL,        max_tokens=32000,        system=SYSTEM_PROMPT,        tools=tools,        thinking={"type": "adaptive"},        output_config={"effort": effort},  # Effort control, but NO format during tool loop        messages=messages    ) as stream:        # Step 2: Handle stream events        current_block_type = None        for event in stream:            if event.type == "content_block_start":                current_block_type = event.content_block.type                if current_block_type == "thinking":                    print("\n[Thinking] ", end="", flush=True)                elif current_block_type == "tool_use":                    print(f"\n[Tool: {event.content_block.name}] ", end="", flush=True)                elif current_block_type == "text":                    print("\n[Response] ", end="", flush=True)                        elif event.type == "content_block_delta":                if hasattr(event.delta, "thinking"):                    print(event.delta.thinking, end="", flush=True)                elif hasattr(event.delta, "text"):                    print(event.delta.text, end="", flush=True)                elif hasattr(event.delta, "partial_json"):                    pass  # Tool input is being constructed                # Step 3: Get the complete response after streaming        response = stream.get_final_message()        # Step 4: Continue the tool loop if needed    while response.stop_reason == "tool_use":        # Execute tools        tool_results = []        for block in response.content:            if block.type == "tool_use":                result = execute_tool(block.name, block.input)                tool_results.append({                    "type": "tool_result",                    "tool_use_id": block.id,                    "content": str(result)                })                print(f"\n  → Result: {result[:150]}..." if len(result) > 150 else f"\n  → Result: {result}")                # Continue with streaming        messages.append({"role": "assistant", "content": response.content})        messages.append({"role": "user", "content": tool_results})                with client.messages.stream(            model=MODEL,            max_tokens=32000,            system=SYSTEM_PROMPT,            tools=tools,            thinking={"type": "adaptive"},            output_config={"effort": effort},            messages=messages        ) as stream:            current_block_type = None            for event in stream:                if event.type == "content_block_start":                    current_block_type = event.content_block.type                    if current_block_type == "thinking":                        print("\n[Thinking] ", end="", flush=True)                    elif current_block_type == "tool_use":                        print(f"\n[Tool: {event.content_block.name}] ", end="", flush=True)                    elif current_block_type == "text":                        print("\n[Response] ", end="", flush=True)                                elif event.type == "content_block_delta":                    if hasattr(event.delta, "thinking"):                        print(event.delta.thinking, end="", flush=True)                    elif hasattr(event.delta, "text"):                        print(event.delta.text, end="", flush=True)                    elif hasattr(event.delta, "partial_json"):                        pass                        response = stream.get_final_message()        # Step 5: After tool loop, make FINAL streamed call for structured output    messages.append({"role": "user", "content": "Provide your structured resolution as JSON."})        with client.messages.stream(        model=MODEL,        max_tokens=8000,        system=SYSTEM_PROMPT,        output_config={"effort": effort, "format": RESOLUTION_SCHEMA},  # Both effort AND format        tool_choice={"type": "none"},        thinking={"type": "adaptive"},        messages=messages    ) as stream:        print("\n[Structured Output] ", end="", flush=True)        for event in stream:            if event.type == "content_block_delta":                if hasattr(event.delta, "text"):                    print(event.delta.text, end="", flush=True)                final = stream.get_final_message()        print()  # New line after streaming        # Step 6: Extract and return structured result    return get_structured_result(final)

## Cell 8: Full Demo

SecureNet Systems just submitted a critical ticket — their admin account is locked out and 47 team members are blocked. Run the full agent with streaming to see all four features composing in real-time: the agentic loop orchestrates tool calls, structured output guarantees the resolution format, adaptive thinking reasons through the security implications, and streaming shows it all as it happens.

In [ ]:
print("Full Agent Demo: Resolving TKT-1045 (account lockout)")
print("   Streaming + Adaptive Thinking + Tools + Structured Output")
print("=" * 60)

start = time.time()
result = run_agent_streaming("Resolve ticket TKT-1045")
elapsed = time.time() - start
if _not_built_yet("run_agent_streaming", result):
    raise SystemExit(0)

print(f"\n\n{'=' * 60}")
print(f"Total time: {elapsed:.1f}s")
print(f"\nStructured Resolution:")
print(json.dumps(result, indent=2))

---
### ✅ CHECKPOINT 3 — Real-time agent

All four features composing in a single agent run:
agentic loop + structured output + adaptive thinking + streaming.

---
# Extra Credit

1. **Tool choice controls** — Use `tool_choice: {"type": "none"}` to force Claude to stop calling tools
2. **Effort optimization** — Process all tickets at `effort="high"`, `effort="medium"`, `effort="low"`. Build a quality/speed table.
3. **Batch processing** — Use the Batch API to process all tickets at once (50% cost reduction).

## Learn More

- [Claude API Documentation](https://docs.anthropic.com/en/docs)
- [Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Adaptive Thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)
- [Structured Outputs](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)
- [Streaming](https://docs.anthropic.com/en/docs/build-with-claude/streaming)